In [ ]:

import pandas as pd

# 加载训练数据
train_file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/monster/train.csv'
train_data = pd.read_csv(train_file_path)

# 显示数据集的前几行
train_data.head()


Displayed on the user's machine.

In [ ]:


# 检查缺失值
print(train_data.isnull().sum())

# 检查数据类型
print(train_data.dtypes)

# 检查类别特征的唯一值
print(train_data['color'].unique())
print(train_data['type'].unique())

# 转换目标变量为数值标签
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_data['type'] = label_encoder.fit_transform(train_data['type'])

# 查看转换后的目标变量
train_data[['type']].head()


id               0
bone_length      0
rotting_flesh    0
hair_length      0
color            0
type             0
dtype: int64
id                 int64
bone_length      float64
rotting_flesh    float64
hair_length      float64
color             object
type              object
dtype: object
['white' 'clear' 'blue' 'black' 'green' 'blood']
['Ghoul' 'Goblin' 'Ghost']


In [ ]:


# 检查缺失值
print(train_data.isnull().sum())

# 检查数据类型
print(train_data.dtypes)

# 检查类别特征的唯一值
print(train_data['color'].unique())
print(train_data['type'].unique())

# 转换目标变量为数值标签
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
train_data['type'] = label_encoder.fit_transform(train_data['type'])

# 查看转换后的目标变量
train_data[['type']].head()


Displayed on the user's machine.

In [ ]:


# 对类别特征 'color' 进行编码
train_data = pd.get_dummies(train_data, columns=['color'])

# 拆分为特征和目标变量
X = train_data.drop(columns=['id', 'type'])
y = train_data['type']

# 拆分为训练集和验证集
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 查看拆分后的数据
X_train.head(), y_train.head()


(     bone_length  rotting_flesh  ...  color_green  color_white
 63      0.431685       0.438959  ...        False        False
 17      0.406880       0.464490  ...        False         True
 215     0.320364       0.212538  ...        False         True
 219     0.342585       0.261752  ...        False        False
 183     0.521237       0.608728  ...        False         True
 
 [5 rows x 9 columns],
 63     2
 17     2
 215    2
 219    2
 183    1
 Name: type, dtype: int64)

In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 选择模型
model = LogisticRegression()

# 训练模型
model.fit(X_train, y_train)

# 预测验证集
y_pred = model.predict(X_val)

# 计算准确率
accuracy = accuracy_score(y_val, y_pred)
accuracy


0.6

In [ ]:



from sklearn.ensemble import RandomForestClassifier

# 选择模型
model = RandomForestClassifier(random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 预测验证集
y_pred = model.predict(X_val)

# 计算准确率
accuracy = accuracy_score(y_val, y_pred)
accuracy



0.6

In [ ]:



from xgboost import XGBClassifier

# 选择模型
model = XGBClassifier(random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 预测验证集
y_pred = model.predict(X_val)

# 计算准确率
accuracy = accuracy_score(y_val, y_pred)
accuracy



0.5833333333333334

In [ ]:



# 特征工程：创建新的特征
train_data['bone_length_times_rotting_flesh'] = train_data['bone_length'] * train_data['rotting_flesh']
train_data['bone_length_times_hair_length'] = train_data['bone_length'] * train_data['hair_length']

# 重新拆分为特征和目标变量
X = train_data.drop(columns=['id', 'type'])
y = train_data['type']

# 重新拆分为训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 选择模型
model = XGBClassifier(random_state=42)

# 训练模型
model.fit(X_train, y_train)

# 预测验证集
y_pred = model.predict(X_val)

# 计算准确率
accuracy = accuracy_score(y_val, y_pred)
accuracy



0.6833333333333333

In [ ]:



from sklearn.model_selection import cross_val_score

# 交叉验证
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_scores, cv_scores.mean()


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

             prefer_skip_nested_validation or global_skip_validation
    214         )
    215     ):
--> 216         return func(*args, **kwargs)
    217 except InvalidParameterError as e:
    218     # When the function is just a wrapper around an estimator, we allow
    219     # the function to delegate validation to the estimator, but we replace
    220     # the name of the estimator by the name of the function in the error
    221     # message to avoid confusion.
    222     msg = re.sub(
    223         r"parameter of \w+ must be",
    224         f"parameter of {func.__qualname__} must be",
    225         str(e),
    226     )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\model_selection\_validation.py:347, in cross_validate(estimator, X, y, groups, scoring, cv,

In [ ]:



import xgboost as xgb

# 创建DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

# 设置XGBoost参数
params = {
    'objective': 'multi:softmax',  # 用于多分类
    'num_class': 3,  # 类别数量
    'eval_metric': 'merror'  # 评估指标
}

# 进行交叉验证
cv_results = xgb.cv(params, dtrain, num_boost_round=100, nfold=5, metrics='merror', early_stopping_rounds=10)

# 查看交叉验证结果
cv_results


Displayed on the user's machine.

In [ ]:



# 计算验证集上的平均准确率
mean_test_merror = cv_results['test-merror-mean'].mean()
mean_test_accuracy = 1 - mean_test_merror
mean_test_accuracy


np.float64(0.691655585106383)